# 📝 과제 02: 온라인 쇼핑 상담 API 자동 배포

<div class="day78-callout" style="margin:10px 0;padding:8px 14px;font-size:17px;line-height:1.75;color:#202124;background:#ffffff;border-left:4px solid #54769b;border-radius:4px;"><strong style="display:block;font-size:1.13em;color:#1f3a5f;margin-bottom:2px;">실습 환경</strong><strong style="font-size:1em;color:#1f3a5f;">내 PC:</strong> 과제 01에서 홈 폴더로 복사한 저장소 폴더(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/shop-deploy</code>)에서 워크플로를 채우고 commit, push 합니다. 이 노트북은 GitHub와 VM에 요청을 보내 확인하고 자가채점합니다.<br><strong style="font-size:1em;color:#1f3a5f;">GitHub:</strong> Actions가 빌드, 테스트, 배포를 차례로 실행하고, 비밀 변수와 변수로 VM 접속 정보를 받습니다.<br><strong style="font-size:1em;color:#1f3a5f;">GCP VM:</strong> 과제 01에서 compose로 띄운 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-api</code>(8081 포트)가 배포 대상입니다. VM에서 직접 실행할 명령은 없습니다.<br><strong style="font-size:1em;color:#1f3a5f;">RunPod:</strong> 이 과제에서는 GPU 파드를 쓰지 않습니다(과제 01 3번 끝에서 종료했습니다).<br><strong style="font-size:1em;color:#1f3a5f;">순서:</strong> 과제 01과 교안 03을 마친 뒤 풉니다.</div>

## 만들 결과

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 03의 배포 파이프라인을 과제 01의 쇼핑 상담 API 저장소에 적용합니다. main에 push하면 ci 워크플로가 코드를 단위 테스트한 뒤 러너에서 이미지를 빌드해 컨테이너로 띄워 테스트하고, 테스트를 통과한 이미지만 GHCR에 올립니다. ci가 성공하면 cd 워크플로가 VM에 배포합니다.</p>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/41_gwaje02_flow.png" width="1050" alt="main에 push하면 ci 워크플로의 unit 작업이 코드를 테스트하고, image 작업이 이미지를 빌드해 컨테이너로 띄워 테스트한 뒤 통과한 이미지만 GHCR에 올립니다. ci가 성공하면 cd 워크플로의 deploy 작업이 SSH로 VM의 compose를 바꿉니다. 테스트가 실패하면 이미지를 올리지 않고 deploy도 건너뜁니다.">

<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">만들 것</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">내 저장소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">workflows/ci.yml</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">workflows/cd.yml</code>에서 TODO (1)~(4)를 채워 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.github/workflows/</code>로 복사한 두 워크플로(unit, image가 성공한 ci 실행, 이어서 deploy가 성공한 cd 실행과, 배포 뒤 VM에서 응답하는 쇼핑 상담 API)</li>
</ul>
</div>

## 준비

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">저장소:</strong> 과제 01의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-deploy</code> 저장소입니다. 내 PC에서는 홈 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/shop-deploy</code>를 VS Code(File &gt; Open Folder)로 열어 고칩니다. 이 과제는 노트북만 있어 일차 폴더에 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">VM:</strong> 과제 01 3번의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-api</code> 컨테이너가 VM의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/shop-deploy/deploy</code>에서 떠 있고, 같은 폴더에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>가 있어야 합니다. 배포 스크립트 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">scripts/deploy_vm.sh</code>가 이 폴더에서 compose로 새 이미지를 받습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">작성할 곳:</strong> 내 저장소(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/shop-deploy</code>)의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">workflows/ci.yml</code>(TODO (1), (2))과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">workflows/cd.yml</code>(TODO (3), (4))에 있는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;"># TODO (n)</code> 주석 아래입니다. 과제 01에서 템플릿을 복사할 때 함께 들어온 빈칸판입니다. 요구사항 번호 (n)이 그 TODO 번호입니다. 다 채운 뒤 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.github/workflows/</code>로 복사해 켭니다. 실습자료 안의 원본은 고치지 않습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배포 키:</strong> 교안 03 3-3에서 만든 배포 키 쌍을 그대로 씁니다. 그 공개 키가 이미 VM의 SSH 키(메타데이터)에 있으므로 VM에서 할 일은 없습니다. 비밀 변수와 변수는 저장소마다 따로라서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-deploy</code> 저장소에 다시 등록합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">자가채점:</strong> GitHub API로 실행 결과를 읽으므로, Actions 화면에서 실행이 끝난 뒤(초록 체크나 빨간 X) 자가채점 셀을 실행합니다.</li>
</ul>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">GitHub 화면에서 할 일</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">비밀 변수(Secrets 탭)에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VM_SSH_KEY</code>로 배포용 개인 키 파일(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/.ssh/serving_deploy</code>) 내용 전체를, 변수(Variables 탭)에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VM_HOST</code>(VM 외부 IP)와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VM_USER</code>(VM 사용자 이름)를 교안 03 3-4와 같은 값으로 등록합니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">누를 곳:</strong> 저장소 위쪽 Settings &gt; Secrets and variables &gt; Actions &gt; Secrets 탭 &gt; New repository secret</li>
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/&lt;GitHub 아이디&gt;/shop-deploy/settings/secrets/actions</code></li>
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">누를 곳:</strong> 같은 화면 Variables 탭 &gt; New repository variable</li>
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/&lt;GitHub 아이디&gt;/shop-deploy/settings/variables/actions</code></li>
</ul>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# ---- 라이브러리, 경로, 주소 ----
import os
from pathlib import Path

import httpx
from dotenv import load_dotenv

day_dir = Path(".").resolve()       # 일차 폴더(day78_모델배포_Docker_CICD)

# 키와 주소는 실습자료/.env 에 모아 둡니다(화면에 출력하지 않습니다)
env_path = day_dir.parent / ".env"
load_dotenv(env_path, override=True)

# 학생 저장소(GitHub API, 올린 파일 원문)와 VM 의 쇼핑 상담 API 주소
GITHUB_OWNER = os.environ["GITHUB_OWNER"]
# GitHub API 요청에 붙일 헤더. 로그인 없이 부르면 같은 와이파이의 교실 전체가 IP 하나의 한도(시간당 60번)를 나눠 쓰므로 읽기 전용 토큰을 붙입니다(학생마다 시간당 5000번)
GITHUB_HEADERS = {"Authorization": "Bearer " + os.environ["GITHUB_READ_TOKEN"], "Accept": "application/vnd.github+json"}  # 교안 00 5-2 의 읽기 전용 토큰(값은 출력하지 않습니다)
REPO_API = "https://api.github.com/repos/" + GITHUB_OWNER + "/shop-deploy"
RAW_URL = "https://raw.githubusercontent.com/" + GITHUB_OWNER + "/shop-deploy/main"
SHOP_URL = "http://" + os.environ["VM_HOST"] + ":8081"

# ci, cd 워크플로의 실행 목록 주소(워크플로 파일 이름으로 찾습니다)
CI_RUNS_URL = REPO_API + "/actions/workflows/ci.yml/runs"
CD_RUNS_URL = REPO_API + "/actions/workflows/cd.yml/runs"
print("저장소:", REPO_API)

## 1. 빌드, 테스트, 배포 워크플로

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배경:</strong> main에 push할 때마다 사람이 VM에 들어가지 않아도 새 이미지가 배포되게 하되, 테스트를 통과한 이미지만 배포해야 합니다. 그래서 커밋마다 도는 테스트와 빌드는 ci 워크플로(unit, image 두 작업을 needs로 잇습니다)에 두고, 배포는 ci가 성공한 main push에서만 도는 cd 워크플로에 따로 둡니다. image 작업은 러너에서 이미지를 빌드해 컨테이너로 띄워 테스트하고, 테스트를 통과한 이미지만 GHCR에 올립니다. 테스트하지 않은 이미지는 GHCR에 올라가지 않습니다. VM 접속 정보는 파일에 쓰지 않고 비밀 변수와 변수로 넘깁니다.</p>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ci.yml</code> 맨 앞의 unit 작업(이미지 없이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tests/test_unit.py</code>로 코드의 함수와 양식을 확인)과 image 작업의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">needs: unit</code>은 완성해 두었습니다. unit이 실패하면 image 작업은 돌지 않습니다. image 작업의 서버 대기, 이미지 테스트(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tests/test_api.py</code>), 실패하면 컨테이너 로그 보기, GHCR 로그인, 이미지 올리기 단계도 완성해 두었습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong> (<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ci.yml</code>의 TODO (1), (2)와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cd.yml</code>의 TODO (3), (4)를 채웁니다. (5)는 아래 터미널 명령에 들어 있습니다)</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ci.yml</code> image 작업의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;"># TODO (1)</code> 자리(빌드 도구 준비 다음)에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker/build-push-action@v7</code> 단계를 둡니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">context</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">load</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">true</code>(빌드한 이미지를 러너의 Docker에 넣어 다음 단계에서 띄웁니다), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">push</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">false</code>(테스트 전이라 GHCR에 올리지 않습니다), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tags</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&#36;{{ steps.image.outputs.name }}:&#36;{{ github.sha }}</code> 한 줄(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">latest</code> 없음), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cache-from</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">type=gha</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cache-to</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">type=gha,mode=max</code>입니다.</li>
<li style="margin:3px 0;">(2) 바로 아래 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;"># TODO (2)</code> 자리에 방금 빌드한 이미지를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker run</code>으로 띄우는 단계를 둡니다. 서비스 컨테이너(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">services</code>)는 쓰지 않습니다.<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">단계의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">env</code> 네 개: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">IMAGE</code>는 (1)의 태그와 같은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&#36;{{ steps.image.outputs.name }}:&#36;{{ github.sha }}</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_BASE_URL</code>은 아무도 듣지 않는 주소 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://127.0.0.1:9/v1</code>(상담 요청이 502가 되어야 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tests/test_api.py</code>의 마지막 테스트가 통과합니다), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>는 CI 전용 아무 글자, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_TIMEOUT_SECONDS</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;5&quot;</code></li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker run</code> 옵션: 백그라운드 실행 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-d</code>, 컨테이너 이름 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--name shop-api</code>(제공한 로그 보기 단계가 이 이름을 씁니다), 포트 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-p 8080:8080</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_</code> 환경 변수 세 개는 값 없이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-e 이름</code>으로 하나씩 넘기고, 맨 끝에 이미지 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;&#36;IMAGE&quot;</code></li>
</ul>
</li>
<li style="margin:3px 0;">(3) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cd.yml</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">on</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">workflow_run</code>으로 ci 워크플로의 실행이 끝났을 때 돕니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">workflows</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[ci]</code>(ci.yml의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">name</code>), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">types</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[completed]</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">branches</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[main]</code>입니다. deploy 작업은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">if</code>로 ci 실행이 성공했고(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">github.event.workflow_run.conclusion</code>이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">'success'</code>) push로 시작된 실행(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">github.event.workflow_run.event</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">'push'</code>)에서만 돕니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">concurrency</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">group</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vm-deploy</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cancel-in-progress</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">false</code>입니다.</li>
<li style="margin:3px 0;">(4) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cd.yml</code> 배포 단계(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">bash scripts/deploy_vm.sh</code>)의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">env</code>에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VM_SSH_KEY</code>는 비밀 변수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&#36;{{ secrets.VM_SSH_KEY }}</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VM_HOST</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VM_USER</code>는 변수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&#36;{{ vars.VM_HOST }}</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&#36;{{ vars.VM_USER }}</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">IMAGE_TAG</code>는 ci 실행의 커밋 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&#36;{{ github.event.workflow_run.head_sha }}</code>를 넘깁니다. cd 실행에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">github.sha</code>는 ci가 테스트한 커밋이 아니라 그때 main의 마지막 커밋입니다.</li>
<li style="margin:3px 0;">(5) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.github/workflows/image.yml</code>은 지워 main에 없습니다.</li>
</ul>
</div>

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">빈칸이 있는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ci.yml</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cd.yml</code>은 내 저장소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">workflows/</code>에 있습니다. GitHub은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.github/workflows/</code> 안의 워크플로만 실행하므로, 빈칸을 채우는 동안에는 실행되지 않습니다. 두 파일의 빈칸을 다 채우기 전에는 커밋하지 않습니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · ~/shop-deploy/workflows/ci.yml</strong></div>

```yaml
# 온라인 쇼핑 상담 API 의 단위 테스트, 이미지 빌드와 테스트, 올리기 워크플로(과제 02). 이 파일과 cd.yml 을 저장소의 .github/workflows/ 로 복사하고, 같은 빌드를 하는 .github/workflows/image.yml 은 지운다
name: ci

on:
  push:
    branches: [main]
  pull_request:
  workflow_dispatch:

permissions:
  contents: read
  packages: write

# 실행할 작업 목록(작업 둘: unit, image)
jobs:
  # 1) 단위 테스트: 이미지 없이 코드의 함수와 양식만 빠르게 확인한다
  unit:
    runs-on: ubuntu-latest
    timeout-minutes: 10
    # 서빙 API 설정(shop_api/core/config.py)이 불러올 때 꼭 읽는 키 자리(테스트 전용 값)
    env:
      VLLM_API_KEY: ci-only-not-a-real-key            # vLLM 키 자리에 넣는 테스트 전용 값(실제 키가 아님)
    steps:
      - name: 저장소 받기
        uses: actions/checkout@v7
      - name: uv 설치
        uses: astral-sh/setup-uv@v10.3.3
        with:
          python-version: "3.12"
          enable-cache: true
      - name: 테스트 라이브러리 설치
        run: uv sync --locked
      - name: 단위 테스트
        run: uv run pytest -v tests/test_unit.py

  # 2) 이미지: 러너에서 이미지를 빌드해 컨테이너로 띄워 테스트하고, 테스트를 통과한 이미지만 GHCR 에 올린다
  image:
    needs: unit                                       # unit 작업이 성공한 뒤에 돈다
    runs-on: ubuntu-latest
    timeout-minutes: 20
    steps:
      - name: 저장소 받기
        uses: actions/checkout@v7
      - name: 이미지 이름 정하기
        id: image
        run: echo "name=ghcr.io/${GITHUB_REPOSITORY,,}" >> "$GITHUB_OUTPUT"
      - name: 빌드 도구(buildx) 준비
        uses: docker/setup-buildx-action@v4

      # TODO (1): build-push-action 으로 이미지를 빌드해 러너의 Docker 에만 두세요(GHCR 에 올리지 않음, 커밋 sha 태그 하나, gha 캐시)

      # TODO (2): 방금 빌드한 이미지를 docker run 으로 컨테이너 shop-api 로 띄우세요(8080 포트, 환경 변수 세 개는 단계의 env 에서 넘김)

      - name: 서버가 뜰 때까지 기다리기
        run: |
          for attempt in $(seq 1 30); do
            curl -fsS http://localhost:8080/healthz && exit 0
            sleep 2
          done
          exit 1
      - name: uv 설치
        uses: astral-sh/setup-uv@v10.3.3
        with:
          python-version: "3.12"
          enable-cache: true
      - name: 테스트 라이브러리 설치
        run: uv sync --locked
      - name: 이미지 테스트
        env:
          SERVING_URL: http://localhost:8080
        run: uv run pytest -v tests/test_api.py
      - name: 실패하면 컨테이너 로그 보기
        if: failure()                                 # 앞 단계 가운데 하나라도 실패했을 때만 실행한다
        run: docker logs shop-api || true             # 빌드 단계에서 멈춰 컨테이너가 없어도 이 단계는 오류로 끝나지 않게 한다

      # 테스트를 통과했을 때만 여기까지 온다
      - name: GHCR 로그인
        if: github.event_name != 'pull_request'       # pull request 실행은 테스트까지만 하고 올리지 않는다
        uses: docker/login-action@v4
        with:
          registry: ghcr.io
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}
      - name: 이미지 올리기
        if: github.event_name != 'pull_request'       # pull request 실행은 올리지 않는다
        env:
          IMAGE: ${{ steps.image.outputs.name }}:${{ github.sha }}   # 올릴 이미지(테스트한 것)
        run: docker push "$IMAGE"
```

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · ~/shop-deploy/workflows/cd.yml</strong></div>

```yaml
# 온라인 쇼핑 상담 API 의 배포 워크플로(과제 02). ci 가 성공한 main 커밋만 VM 에 배포한다. 이 파일을 ci.yml 과 함께 저장소의 .github/workflows/ 로 복사한다
name: cd

# TODO (3): ci 워크플로의 실행이 끝나면 돌게 하세요(main 브랜치의 실행만)

permissions:
  contents: read

jobs:
  deploy:
    runs-on: ubuntu-latest
    timeout-minutes: 10

    # TODO (3): ci 가 성공했고 main 에 push 해 시작된 실행에서만 돌게 하고, 동시 배포를 막는 묶음 vm-deploy 를 정하세요

    steps:
      - name: 저장소 받기
        uses: actions/checkout@v7
        with:
          ref: ${{ github.event.workflow_run.head_sha }}   # ci 실행의 커밋 sha(ci 가 테스트한 바로 그 커밋을 받는다)
      - name: VM 에 배포하고 /healthz 확인
        run: bash scripts/deploy_vm.sh
        # TODO (4): 비밀 변수 VM_SSH_KEY, 변수 VM_HOST 와 VM_USER, ci 실행의 커밋 sha 를 IMAGE_TAG 로 env 에 넘기세요
```

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · shop-deploy/scripts/deploy_vm.sh(제공)</strong></div>

```bash
#!/usr/bin/env bash
# GitHub Actions 의 deploy 작업이 GCP VM 에 새 이미지를 배포하고 확인하는 스크립트(과제 02)
# 필요한 환경 변수: VM_SSH_KEY(배포용 개인 키), VM_HOST(VM 외부 IP), VM_USER(VM 사용자 이름), IMAGE_TAG(배포할 이미지 태그)
# 명령 하나라도 실패하거나 없는 변수를 쓰면 멈춰, deploy 작업이 실패로 끝난다
set -euo pipefail

# 0) 필요한 값이 비어 있으면 무엇이 빠졌는지 알리고 멈춘다(등록하지 않은 변수는 빈 글자로 들어온다)
: "${VM_SSH_KEY:?비밀 변수 VM_SSH_KEY 가 비었습니다}"
: "${VM_HOST:?변수 VM_HOST 가 비었습니다}"
: "${VM_USER:?변수 VM_USER 가 비었습니다}"

# 태그는 커밋 sha(16진수 40자리)만 받는다. 태그 값이 VM 의 명령 줄에 그대로 들어가므로 다른 글자를 막는다
if [[ ! "$IMAGE_TAG" =~ ^[0-9a-f]{40}$ ]]; then
  echo "IMAGE_TAG 는 커밋 sha 전체(16진수 40자리)여야 합니다: $IMAGE_TAG"
  exit 1
fi

# VM 안의 저장소 폴더, compose 서비스 이름, 확인할 포트
APP_DIR="shop-deploy"
SERVICE="shop-api"
PORT="8081"

# 1) 비밀 변수의 개인 키를 러너의 파일로 저장한다(값은 로그에 찍지 않는다)
mkdir -p ~/.ssh
printf '%s\n' "$VM_SSH_KEY" > ~/.ssh/deploy_key
chmod 600 ~/.ssh/deploy_key   # 소유자만 읽게 한다. ssh 는 다른 사용자도 읽을 수 있는 개인 키를 쓰지 않는다
# 러너는 처음 접속 때 묻는 확인(yes/no)에 답할 수 없어, VM 의 호스트 키를 known_hosts 에 미리 넣는다
if ! ssh-keyscan -H "$VM_HOST" >> ~/.ssh/known_hosts; then
  echo "VM($VM_HOST) 의 22번 포트에서 호스트 키를 받지 못했습니다. VM 이 켜져 있는지, VM_HOST 가 맞는지 봅니다"
  exit 1
fi
SSH="ssh -i $HOME/.ssh/deploy_key $VM_USER@$VM_HOST"

# 2) VM 의 deploy 폴더에서 새 태그의 이미지를 받고 컨테이너를 바꾼다(compose.yaml 의 ${IMAGE_TAG:-latest} 가 이 sha 가 된다)
$SSH "cd ~/$APP_DIR/deploy && IMAGE_TAG=$IMAGE_TAG docker compose pull $SERVICE && IMAGE_TAG=$IMAGE_TAG docker compose up -d $SERVICE"

# 3) 지금 도는 컨테이너가 방금 받은 이미지인지 확인한다(옛 컨테이너가 /healthz 에 답해 성공으로 보이는 것을 막는다)
RUNNING=$($SSH "cd ~/$APP_DIR/deploy && docker inspect --format '{{.Config.Image}}' \$(docker compose ps -q $SERVICE)")
echo "실행 중인 이미지: $RUNNING"
case "$RUNNING" in
  *":$IMAGE_TAG") ;;
  *) echo "실행 중인 컨테이너가 $IMAGE_TAG 이미지가 아닙니다"; exit 1 ;;
esac

# 4) /healthz 가 200 을 돌려줄 때까지 2초마다, 최대 30번 확인한다
for attempt in $(seq 1 30); do
  # --max-time 5: 한 번의 요청이 5초를 넘기면 끊고 다음 시도로 간다
  if curl -fsS --max-time 5 "http://$VM_HOST:$PORT/healthz"; then
    echo
    echo "배포 확인: $IMAGE_TAG 이미지가 /healthz 에 응답합니다"
    exit 0
  fi
  sleep 2
done
echo "/healthz 가 응답하지 않습니다. VM 에서 docker compose logs $SERVICE 로 원인을 봅니다"
exit 1
```

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 교안 03의 2절(CI: ci.yml), 3절(CD: cd.yml)과 교안 01의 5절(GitHub Actions로 이미지 올리기)을 참고합니다.
세부구현
1) 이미지 빌드 단계는 과제 01 image.yml의 빌드 단계에서 push를 false로 바꾸고 load를 더한 모양이고, 태그 줄은 sha 하나로 줄입니다.
2) docker run의 -e 뒤에 값 없이 이름만 쓰면 그 단계 env의 같은 이름 값을 컨테이너에 넘깁니다. 이미지 이름도 단계 env의 IMAGE에 두고 run에서 "$IMAGE"로 읽습니다.
3) if, concurrency는 작업 이름 아래(runs-on과 같은 들여쓰기)에 두고, cd.yml의 on은 name, permissions와 같은 맨 왼쪽에 둡니다.
4) 단계의 env는 그 단계의 run과 같은 들여쓰기에 두고, 비밀 변수는 secrets 문맥, 변수는 vars 문맥에서 읽습니다. cd를 시작한 ci 실행의 정보는 github.event.workflow_run 에 있습니다.
5) YAML은 들여쓰기로 구조를 정하므로 탭 대신 공백 두 칸씩을 씁니다.
```

</details>

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">빈칸을 다 채워 저장한 뒤 올립니다. 아래 블록은 채운 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">workflows/ci.yml</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">workflows/cd.yml</code>을 GitHub이 실행하는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.github/workflows/</code>로 복사하고, 요구사항 (5)대로 과제 01의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.github/workflows/image.yml</code>을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">git rm</code>으로 지운 뒤(같은 push에서 이미지를 두 번 빌드하지 않게) 함께 커밋해 올립니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
pushd ~/shop-deploy

cp workflows/ci.yml .github/workflows/ci.yml
cp workflows/cd.yml .github/workflows/cd.yml

git rm .github/workflows/image.yml

git add workflows/ci.yml workflows/cd.yml .github/workflows/ci.yml .github/workflows/cd.yml
git commit -m "빌드, 테스트 워크플로(ci)와 배포 워크플로(cd) 켜기"
git push
popd
```

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">GitHub 화면에서 할 일</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ci</code> 실행의 그래프에서 unit, image가 차례로 초록 체크가 되는지 봅니다. ci가 끝나면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cd</code>에 새 실행이 생기고, 그 실행의 deploy가 초록 체크가 되는지 봅니다. 실패한 작업은 이름을 눌러 멈춘 단계의 로그를 봅니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">누를 곳:</strong> 저장소 위쪽 Actions &gt; 왼쪽 목록의 ci &gt; 맨 위 실행</li>
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/&lt;GitHub 아이디&gt;/shop-deploy/actions/workflows/ci.yml</code></li>
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">누를 곳:</strong> 같은 화면 왼쪽 목록의 cd &gt; 맨 위 실행</li>
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/&lt;GitHub 아이디&gt;/shop-deploy/actions/workflows/cd.yml</code></li>
</ul>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
import yaml

# [자가채점] 올린 두 워크플로 파일의 구조와, unit, image 가 성공한 ci 실행에 이어 deploy 가 성공한 cd 실행을 확인합니다
ci_response = httpx.get(RAW_URL + "/.github/workflows/ci.yml", timeout=30)
assert ci_response.status_code == 200, f".github/workflows/ci.yml 을 읽지 못했습니다(상태 코드 {ci_response.status_code}). 복사해 push 했는지 확인하세요. push 직후라면 몇 분 뒤 다시 실행하세요."
cd_response = httpx.get(RAW_URL + "/.github/workflows/cd.yml", timeout=30)
assert cd_response.status_code == 200, f".github/workflows/cd.yml 을 읽지 못했습니다(상태 코드 {cd_response.status_code}). 복사해 push 했는지 확인하세요. push 직후라면 몇 분 뒤 다시 실행하세요."
ci_workflow = yaml.safe_load(ci_response.text)
cd_workflow = yaml.safe_load(cd_response.text)
assert isinstance(ci_workflow, dict) and isinstance(cd_workflow, dict), "ci.yml 이나 cd.yml 을 YAML 로 읽지 못했습니다. 들여쓰기를 확인하세요."

# PyYAML 은 YAML 1.1 규칙으로 키 on 을 참(True)으로 읽으므로, 실행 조건은 workflow[True] 에 있습니다
ci_trigger = ci_workflow.get(True) or {}
assert isinstance(ci_trigger, dict) and (ci_trigger.get("push") or {}).get("branches") == ["main"], "ci.yml 의 on.push.branches 가 [main] 이 아닙니다. 제공한 on 부분은 고치지 마세요."
ci_jobs = ci_workflow.get("jobs") or {}
assert list(ci_jobs) == ["unit", "image"], f"ci.yml 의 작업 이름이 {list(ci_jobs)}입니다. 제공한 unit, image 두 작업이어야 합니다."
image_workflow = httpx.get(RAW_URL + "/.github/workflows/image.yml", timeout=30)
assert image_workflow.status_code == 404, "main 에 .github/workflows/image.yml 이 남아 있습니다. git rm 으로 지우고 push 하세요(요구사항 (5)). push 직후라면 몇 분 뒤 다시 실행하세요."


# 공백과 줄바꿈을 지운 글자(${{ a }} 와 ${{a}} 를 같게 봅니다)
def compact(value):
    return "".join(str(value).split())


# 같은 뜻의 긴 옵션 이름을 짧은 이름으로 바꿉니다(--detach 와 -d 를 같게 봅니다)
SHORT_OPTIONS = {"--detach": "-d", "--publish": "-p", "--env": "-e"}


# docker 명령을 띄어쓰기로 나눈 조각 목록(줄 끝의 \ 로 이은 명령도 한 줄로 봅니다)
# 따옴표는 떼고, --name=shop-api 처럼 = 로 붙여 쓴 긴 옵션은 옵션과 값 두 조각으로 나눕니다
def command_words(step):
    words = []
    for word in str(step.get("run", "")).replace("\\\n", " ").split():
        word = word.replace('"', "").replace("'", "")
        if word.startswith("--") and "=" in word:
            option, value = word.split("=", 1)
            words += [SHORT_OPTIONS.get(option, option), value]
        else:
            words.append(SHORT_OPTIONS.get(word, word))
    return words


# 조각 목록에서 옵션(예: -e) 바로 뒤에 온 값들
def values_after(words, option):
    return [words[i + 1] for i in range(len(words) - 1) if words[i] == option]


# 제공한 unit 작업(단위 테스트)이 먼저 돌고, image 작업은 unit 이 성공한 뒤에 돕니다
unit_job = ci_jobs["unit"] or {}
unit_runs = [compact(step.get("run", "")) for step in unit_job.get("steps") or []]
assert "uvrunpytest-vtests/test_unit.py" in unit_runs, "unit 작업이 tests/test_unit.py 를 실행하지 않습니다. 제공한 unit 작업은 고치지 마세요."
image_job = ci_jobs["image"] or {}
assert image_job.get("needs") in ["unit", ["unit"]], f"image 작업의 needs 가 {image_job.get('needs')!r}입니다. 제공한 needs: unit 은 고치지 마세요."
image_steps = [step for step in image_job.get("steps") or [] if isinstance(step, dict)]
built_image = "${{steps.image.outputs.name}}:${{github.sha}}"

# image 작업에서 이미지 빌드, 컨테이너 띄우기, 이미지 테스트, 이미지 올리기 단계의 자리(없으면 None)
build_index = run_index = test_index = push_index = None
for index, step in enumerate(image_steps):
    if compact(step.get("uses", "")).startswith("docker/build-push-action@"):
        build_index = index
    if command_words(step)[:2] == ["docker", "run"]:
        run_index = index
    if compact(step.get("run", "")) == "uvrunpytest-vtests/test_api.py":
        test_index = index
    if command_words(step)[:2] == ["docker", "push"]:
        push_index = index

# 제공한 이미지 테스트 단계와, 테스트를 통과한 뒤에만 sha 태그 이미지를 올리는 이미지 올리기 단계
assert test_index is not None, "image 작업에 tests/test_api.py 를 실행하는 이미지 테스트 단계가 없습니다. 제공한 이미지 테스트 단계는 고치지 마세요."
assert push_index is not None and push_index > test_index, "image 작업의 이미지 올리기(docker push) 단계가 없거나 이미지 테스트 단계보다 앞에 있습니다. 제공한 GHCR 로그인, 이미지 올리기 단계는 고치지 마세요."
push_step = image_steps[push_index]
assert command_words(push_step)[-1] in ["$IMAGE", "${IMAGE}"] and compact((push_step.get("env") or {}).get("IMAGE", "")) == built_image, "이미지 올리기 단계가 커밋 sha 태그 이미지를 올리지 않습니다. 제공한 이미지 올리기 단계는 고치지 마세요."

# (1) 이미지 빌드 단계: 러너의 Docker 에만 두고(load true, push false) 아직 올리지 않습니다
# Actions 는 with 의 값을 글자로 넘기므로 load, push 는 글자로 바꿔 'true' 와 true 를 같게 봅니다
assert build_index is not None, "ci.yml 의 image 작업에 docker/build-push-action 단계가 없습니다(TODO (1))."
build_with = image_steps[build_index].get("with")
assert isinstance(build_with, dict), "build-push-action 단계에 with 가 없습니다(TODO (1))."
assert build_with.get("context") == ".", f"build-push-action 의 context 가 {build_with.get('context')!r}입니다. . 이어야 합니다(TODO (1))."
assert str(build_with.get("load")).lower() == "true", f"build-push-action 의 load 가 {build_with.get('load')!r}입니다. 빌드한 이미지를 러너의 Docker 에 넣도록 true 여야 합니다(TODO (1))."
assert str(build_with.get("push")).lower() == "false", f"build-push-action 의 push 가 {build_with.get('push')!r}입니다. 테스트 전이라 false 여야 합니다(TODO (1))."
assert compact(build_with.get("tags", "")) == built_image, f"tags 가 {build_with.get('tags')!r}입니다. 커밋 sha 태그 하나(latest 없음)만 한 줄로 주세요(TODO (1))."
assert build_with.get("cache-from") == "type=gha" and compact(build_with.get("cache-to", "")) == "type=gha,mode=max", "cache-from 은 type=gha, cache-to 는 type=gha,mode=max 여야 합니다(TODO (1))."

# (2) 컨테이너 띄우기 단계: 서비스 컨테이너 없이 방금 빌드한 이미지를 docker run 으로 띄웁니다
assert "services" not in image_job, "image 작업에 services 가 있습니다. 서비스 컨테이너 대신 docker run 단계로 띄우세요(TODO (2))."
assert run_index is not None, "image 작업에 docker run 으로 컨테이너를 띄우는 단계가 없습니다(TODO (2))."
assert build_index < run_index < test_index, "docker run 단계는 이미지 빌드 단계 뒤, 이미지 테스트 단계 앞에 두세요(TODO (2))."
run_step = image_steps[run_index]
run_env = run_step.get("env") or {}
assert isinstance(run_env, dict), "docker run 단계에 env 가 없습니다(TODO (2))."
assert sorted(run_env) == ["IMAGE", "VLLM_API_KEY", "VLLM_BASE_URL", "VLLM_TIMEOUT_SECONDS"], f"docker run 단계의 env 키가 {sorted(run_env)}입니다. IMAGE 와 VLLM_ 세 개여야 합니다. 쇼핑 상담 API에는 ADMIN_API_KEY가 없습니다(TODO (2))."
assert compact(run_env["IMAGE"]) == built_image, f"docker run 단계 env 의 IMAGE 가 {run_env['IMAGE']!r}입니다. 이미지 빌드 단계의 태그와 같아야 합니다(TODO (2))."
assert run_env["VLLM_BASE_URL"] == "http://127.0.0.1:9/v1", "VLLM_BASE_URL 은 아무도 듣지 않는 주소 http://127.0.0.1:9/v1 이어야 합니다(TODO (2))."
assert str(run_env["VLLM_TIMEOUT_SECONDS"]) == "5", "VLLM_TIMEOUT_SECONDS 는 \"5\" 여야 합니다(TODO (2))."
run_words = command_words(run_step)
assert "-d" in run_words, "docker run 에 백그라운드 실행 -d 가 없습니다(TODO (2))."
assert values_after(run_words, "--name") == ["shop-api"], f"docker run 의 --name 이 {values_after(run_words, '--name')}입니다. shop-api 여야 합니다(TODO (2))."
assert values_after(run_words, "-p") == ["8080:8080"], f"docker run 의 -p 가 {values_after(run_words, '-p')}입니다. 8080:8080 이어야 합니다(TODO (2))."
# -e 이름 과 -e 이름=$이름 을 같게 봅니다(= 앞의 이름만 비교합니다)
passed_env = [value.split("=")[0] for value in values_after(run_words, "-e")]
for name in ["VLLM_BASE_URL", "VLLM_API_KEY", "VLLM_TIMEOUT_SECONDS"]:
    assert name in passed_env, f"docker run 이 -e {name} 으로 환경 변수를 넘기지 않습니다(TODO (2))."
assert run_words[-1] in ["$IMAGE", "${IMAGE}"], f"docker run 의 맨 끝이 {run_words[-1]}입니다. 이미지 \"$IMAGE\" 여야 합니다(TODO (2))."

# (3) cd 의 실행 조건(on.workflow_run)과 deploy 작업의 if, concurrency
cd_trigger = cd_workflow.get(True) or {}
assert isinstance(cd_trigger, dict) and "workflow_run" in cd_trigger, "cd.yml 의 on 에 workflow_run 이 없습니다. workflow_run 으로 ci 워크플로의 실행이 끝났을 때 돌게 하세요(TODO (3))."
workflow_run = cd_trigger["workflow_run"] or {}
assert isinstance(workflow_run, dict), "cd.yml 의 on.workflow_run 아래에 workflows, types, branches 를 두세요(TODO (3))."
assert workflow_run.get("workflows") == ["ci"], f"on.workflow_run.workflows 가 {workflow_run.get('workflows')!r}입니다. ci.yml 의 name 인 ci 하나여야 합니다(TODO (3))."
assert "completed" in (workflow_run.get("types") or []), f"on.workflow_run.types 가 {workflow_run.get('types')!r}입니다. 실행이 끝났을 때(completed) 돌게 하세요(TODO (3))."
assert "main" in (workflow_run.get("branches") or []), f"on.workflow_run.branches 가 {workflow_run.get('branches')!r}입니다. main 브랜치의 실행만 보게 하세요(TODO (3))."
cd_jobs = cd_workflow.get("jobs") or {}
assert list(cd_jobs) == ["deploy"], f"cd.yml 의 작업 이름이 {list(cd_jobs)}입니다. 제공한 deploy 작업 하나여야 합니다."
deploy_job = cd_jobs["deploy"] or {}
deploy_if = compact(deploy_job.get("if", ""))
assert "workflow_run.conclusion=='success'" in deploy_if, f"deploy 작업의 if 가 {deploy_job.get('if')!r}입니다. ci 실행이 성공했을 때만 돌게 하세요(TODO (3))."
assert "workflow_run.event=='push'" in deploy_if, f"deploy 작업의 if 가 {deploy_job.get('if')!r}입니다. push 로 시작된 ci 실행에서만 돌게 하세요(TODO (3))."
assert "&&" in deploy_if and "||" not in deploy_if, f"deploy 작업의 if 가 {deploy_job.get('if')!r}입니다. 두 조건을 && 로 묶어 둘 다 맞을 때만 돌게 하세요(TODO (3))."
deploy_concurrency = deploy_job.get("concurrency") or {}
assert isinstance(deploy_concurrency, dict) and deploy_concurrency.get("group") == "vm-deploy" and deploy_concurrency.get("cancel-in-progress") is False, "deploy 작업의 concurrency 는 group vm-deploy, cancel-in-progress false 여야 합니다(TODO (3))."

# (4) cd 의 배포 단계의 env
deploy_step = None
for step in deploy_job.get("steps") or []:
    if "deploy_vm.sh" in str(step.get("run", "")):
        deploy_step = step
assert deploy_step is not None and isinstance(deploy_step.get("env"), dict), "deploy_vm.sh 를 실행하는 단계에 env 가 없습니다(TODO (4))."
expected_env = {"VM_SSH_KEY": "secrets.VM_SSH_KEY", "VM_HOST": "vars.VM_HOST", "VM_USER": "vars.VM_USER", "IMAGE_TAG": "github.event.workflow_run.head_sha"}
for name, source in expected_env.items():
    assert source in compact(deploy_step["env"].get(name, "")), f"배포 단계 env 의 {name} 이 {source} 를 읽지 않습니다(TODO (4))."

# GitHub API: ci 가 끝나 시작된 cd 실행 가운데 deploy 작업이 성공한 가장 최근 실행(ci 가 실패하거나 push 가 아니면 deploy 는 skipped 입니다)
cd_runs_response = httpx.get(CD_RUNS_URL, params={"event": "workflow_run", "status": "success", "per_page": 20}, headers=GITHUB_HEADERS, timeout=30)
assert cd_runs_response.status_code == 200, f"cd 실행 목록을 읽지 못했습니다(상태 코드 {cd_runs_response.status_code}). .env 의 GITHUB_READ_TOKEN 이 이 저장소를 읽을 수 있는지 확인하세요."
deployed_run = None
for run in cd_runs_response.json()["workflow_runs"]:
    cd_jobs_response = httpx.get(REPO_API + "/actions/runs/" + str(run["id"]) + "/jobs", headers=GITHUB_HEADERS, timeout=30)
    assert cd_jobs_response.status_code == 200, f"cd 실행의 작업 목록을 읽지 못했습니다(상태 코드 {cd_jobs_response.status_code}). .env 의 GITHUB_READ_TOKEN 을 확인하세요."
    if [(job["name"], job["conclusion"]) for job in cd_jobs_response.json()["jobs"]] == [("deploy", "success")]:
        deployed_run = run
        break
assert deployed_run is not None, "deploy 작업이 성공한 cd 실행이 없습니다. 실행 중이면 Actions 화면에서 cd 실행이 끝난 뒤 다시 실행하세요. 끝났다면 ci 가 성공했는지, cd 의 deploy 가 skipped 나 실패인지 확인하세요. 배포에서 멈췄다면 비밀 변수와 변수 이름, VM 의 ~/shop-deploy/deploy/.env 를 확인하세요."

# 같은 커밋(head_sha)을 main 에 push 해 성공한 ci 실행의 작업 결과
ci_runs_response = httpx.get(CI_RUNS_URL, params={"head_sha": deployed_run["head_sha"], "event": "push", "status": "success", "per_page": 1}, headers=GITHUB_HEADERS, timeout=30)
assert ci_runs_response.status_code == 200, f"ci 실행 목록을 읽지 못했습니다(상태 코드 {ci_runs_response.status_code}). .env 의 GITHUB_READ_TOKEN 을 확인하세요."
ci_runs = ci_runs_response.json()
assert ci_runs["total_count"] >= 1, f"cd 가 배포한 커밋 {deployed_run['head_sha'][:7]} 을 main 에 push 해 성공한 ci 실행이 없습니다. Actions 화면의 ci 목록을 확인하세요."
ci_jobs_response = httpx.get(REPO_API + "/actions/runs/" + str(ci_runs["workflow_runs"][0]["id"]) + "/jobs", headers=GITHUB_HEADERS, timeout=30)
assert ci_jobs_response.status_code == 200, f"ci 실행의 작업 목록을 읽지 못했습니다(상태 코드 {ci_jobs_response.status_code}). .env 의 GITHUB_READ_TOKEN 을 확인하세요."
job_results = {}
for job in ci_jobs_response.json()["jobs"]:
    job_results[job["name"]] = job["conclusion"]
assert job_results == {"unit": "success", "image": "success"}, f"배포한 커밋의 ci 작업 결과가 {job_results}입니다. unit, image 가 모두 success 여야 합니다."

# VM 의 쇼핑 상담 API 가 배포 뒤에도 응답합니다
health = httpx.get(SHOP_URL + "/healthz", timeout=30)
assert health.status_code == 200, f"VM 의 /healthz 가 {health.status_code}입니다. Actions 의 마지막 cd 실행 로그와 VM 의 docker compose ps 를 확인하세요."
print("✅ 통과!")

## 마무리

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이 과제가 끝나면 VM의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-api</code>는 마지막으로 성공한 배포의 sha 태그 이미지로 떠 있습니다. 두 저장소(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-deploy</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-deploy</code>)에는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ci.yml</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cd.yml</code>만 켜져 있고 되돌리기 워크플로(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">rollback.yml</code>)는 아직 없습니다. 다음 단원에서 이 저장소와 VM을 그대로 씁니다. 과정을 모두 마치면 GPU 파드를 아직 종료하지 않았다면 종료하고(과제 01 3번 끝), 부록 01(GCP VM 준비) 10절의 순서로 VM을 삭제합니다. GHCR 패키지 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-deploy</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-deploy</code>도 둘 다 지웁니다(GitHub 프로필 &gt; Packages &gt; 패키지 &gt; Package settings &gt; Delete this package).</p>
</div>